In [1]:
library(readxl)
library(dplyr)
library(tidyr)
library(stringr)
library(psych)      # PCA / fator
library(scales)
library(lavaan)
library(semTools)
library(tidyverse)

Warning message:
"pacote 'readxl' foi compilado no R versão 4.5.3"

Anexando pacote: 'dplyr'


Os seguintes objetos são mascarados por 'package:stats':

    filter, lag


Os seguintes objetos são mascarados por 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"pacote 'psych' foi compilado no R versão 4.5.3"

Anexando pacote: 'scales'


Os seguintes objetos são mascarados por 'package:psych':

    alpha, rescale


This is lavaan 0.6-21
lavaan is FREE software! Please report any bugs.


Anexando pacote: 'lavaan'


O seguinte objeto é mascarado por 'package:psych':

    cor2cov


 

###############################################################################

This is semTools 0.5-7

All users of R (or SEM) are invited to submit functions or ideas for functions.

###############################################################################


Anexando pacote: 'semTools'


Os seguintes objetos são mascarados por 'package:psych':

    reliability, skew


Warnin

In [10]:
install.packages("janitor")
library(janitor)

Instalando pacote em 'C:/Users/Maxwell/AppData/Local/R/win-library/4.5'
(como 'lib' não foi especificado)

instalando a dependência 'snakecase' também




pacote 'snakecase' desempacotado com sucesso e somas MD5 verificadas
pacote 'janitor' desempacotado com sucesso e somas MD5 verificadas

Os pacotes binários baixados estão em
	C:\Users\Maxwell\AppData\Local\Temp\RtmpcDkzNY\downloaded_packages


Warning message:
"pacote 'janitor' foi compilado no R versão 4.5.3"

Anexando pacote: 'janitor'


Os seguintes objetos são mascarados por 'package:stats':

    chisq.test, fisher.test




In [38]:
knowledge <- read_excel("Knowledge.xlsx")
skills <- read_excel("Skills.xlsx")
activities <- read_excel("Work_Activities.xlsx")
job <- read_excel("Job_Zones.xlsx")
context <- read_excel("Work_Context.xlsx")

In [43]:
clean_onet <- function(df) {
  df %>%
    clean_names() %>%
    filter(recommend_suppress != "Y") %>%
    filter(scale_id == "IM") %>%
    select(o_net_soc_code, element_name, data_value) %>%
    group_by(o_net_soc_code, element_name) %>%
    summarise(value = mean(data_value, na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = element_name, values_from = value)
}

clean_context <- function(df) {
  df %>%
    clean_names() %>%
    filter(recommend_suppress != "Y") %>%
    filter(scale_id %in% c("CX")) %>%   # 👈 chave aqui
    select(o_net_soc_code, element_name, data_value) %>%
    group_by(o_net_soc_code, element_name) %>%
    summarise(value = mean(data_value, na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = element_name, values_from = value)
}

In [44]:
knowledge_w  <- clean_onet(knowledge)
skills_w     <- clean_onet(skills)
activities_w <- clean_onet(activities)
context_w    <- clean_context(context)

In [45]:
prepare_matrix <- function(df) {
  mat <- df %>%
    select(-o_net_soc_code)
  
  # permitir mais NA (contexto é mais incompleto)
  mat <- mat %>%
    select(where(~ mean(is.na(.)) < 0.6))
  
  # imputar média
  mat <- mat %>%
    mutate(across(everything(), ~ ifelse(is.na(.), mean(., na.rm = TRUE), .)))
  
  # filtro de variância mais flexível
  mat <- mat %>%
    select(where(~ sd(., na.rm = TRUE) > 0.001))
  
  return(as.matrix(mat))
}

In [46]:
run_pca <- function(mat) {
  psych::principal(mat, nfactors = 1, rotate = "none", scores = TRUE)
}

In [48]:
dim(mat_knowledge)
dim(mat_skills)
dim(mat_activities)
dim(mat_context)

[1] 683  33

[1] 894  35

[1] 683  41

[1] 683  55

In [47]:
# preparar matrizes
mat_knowledge  <- prepare_matrix(knowledge_w)
mat_skills     <- prepare_matrix(skills_w)
mat_activities <- prepare_matrix(activities_w)
mat_context    <- prepare_matrix(context_w)

# PCA
pca_knowledge  <- run_pca(mat_knowledge)
pca_skills     <- run_pca(mat_skills)
pca_activities <- run_pca(mat_activities)
pca_context    <- run_pca(mat_context)

In [49]:
index_knowledge  <- pca_knowledge$scores[,1]
index_skills     <- pca_skills$scores[,1]
index_activities <- pca_activities$scores[,1]
index_context    <- pca_context$scores[,1]

In [52]:
df_knowledge <- knowledge_w %>%
  select(o_net_soc_code) %>%
  mutate(knowledge = index_knowledge)

df_skills <- skills_w %>%
  select(o_net_soc_code) %>%
  mutate(skills = index_skills)

df_activities <- activities_w %>%
  select(o_net_soc_code) %>%
  mutate(activities = index_activities)

df_context <- context_w %>%
  select(o_net_soc_code) %>%
  mutate(context = index_context)

In [53]:
df_index <- df_knowledge %>%
  inner_join(df_skills, by = "o_net_soc_code") %>%
  inner_join(df_activities, by = "o_net_soc_code") %>%
  inner_join(df_context, by = "o_net_soc_code")

In [54]:
dim(df_index)

[1] 683   5

In [55]:
mat_final <- df_index %>%
  select(knowledge, skills, activities, context) %>%
  as.matrix()

pca_final <- psych::principal(mat_final, nfactors = 1, rotate = "none", scores = TRUE)

index_qualification <- pca_final$scores[,1]

df_index$qualification <- index_qualification

In [56]:
pca_final$Vaccounted

,PC1
SS loadings,2.9418989
Proportion Var,0.7354747


In [57]:
pca_final$loadings


Loadings:
           PC1   
knowledge   0.885
skills      0.941
activities  0.822
context    -0.773

                 PC1
SS loadings    2.942
Proportion Var 0.735

In [58]:
cor(df_index$qualification, df_index$knowledge)

[1] 0.8845291